# Kapitel 5: Was hat die Maschine gelernt?

## Die Reise bis hierher

Wir begannen mit einer Frage: *Kann eine Maschine menschliche Emotionen aus Text erkennen?*

- **Kapitel 1** — Wir entdeckten 400.000 Amazon-Bewertungen, perfekt balanciert
- **Kapitel 2** — Wir bereinigten die Daten und schufen klare Sentiment-Labels
- **Kapitel 3** — Wir verwandelten Wörter in TF-IDF-Vektoren
- **Kapitel 4** — Ein Logistic-Regression-Modell erreichte 84,7% Accuracy

## Dieses Kapitel

Jetzt erzählen wir die Geschichte visuell. Jede Grafik beantwortet eine Frage —
und zusammen ergeben sie das Gesamtbild unseres Projekts.

## 5.1 Setup und Daten laden

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviews – Visualisierung") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

df_clean = spark.read.parquet(
    "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/cleaned_reviews.parquet"
)
df_preds = spark.read.parquet(
    "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/predictions.parquet"
)

print(f"Bereinigte Daten: {df_clean.count():,} Zeilen")
print(f"Vorhersagen:      {df_preds.count():,} Zeilen")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold'
})

## 5.2 Frage 1: Wie sieht unser Datensatz aus?

In [ ]:
from pyspark.sql.functions import col, count as spark_count

label_dist = df_clean.groupBy("label").agg(spark_count("*").alias("count")).orderBy("label").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ["#F09595", "#5DCAA5"]
edge_colors = ["#A32D2D", "#0F6E56"]
bar_labels = ["Negativ (0)", "Positiv (1)"]
bars = axes[0].bar(bar_labels, label_dist["count"], color=colors, edgecolor=edge_colors)
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., h, f'{int(h):,}', ha='center', va='bottom', fontsize=11)
axes[0].set_title("Sentiment-Verteilung")
axes[0].set_ylabel("Anzahl")

axes[1].pie(label_dist["count"], labels=bar_labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12}, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title("Anteil positiv / negativ")

plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/01_sentiment_verteilung.png", dpi=150, bbox_inches='tight')
plt.show()

**Antwort:** Perfekte 50/50-Balance. Das Modell hatte faire Bedingungen zum Lernen.

## 5.3 Frage 2: Schreiben zufriedene und unzufriedene Kunden unterschiedlich lang?

In [ ]:
from pyspark.sql.functions import length

df_len = df_clean.withColumn("text_length", length(col("text")))
pos_len = df_len.filter(col("label") == 1).select("text_length").toPandas()
neg_len = df_len.filter(col("label") == 0).select("text_length").toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(neg_len["text_length"].dropna(), bins=50, alpha=0.6, color="#F09595", edgecolor="#A32D2D", label="Negativ", range=(0, 3000))
ax.hist(pos_len["text_length"].dropna(), bins=50, alpha=0.6, color="#5DCAA5", edgecolor="#0F6E56", label="Positiv", range=(0, 3000))
ax.set_xlabel("Textlänge (Zeichen)")
ax.set_ylabel("Anzahl")
ax.set_title("Textlängen-Verteilung nach Sentiment")
ax.legend()

plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/02_textlaenge_sentiment.png", dpi=150, bbox_inches='tight')
plt.show()

**Antwort:** Positive Reviews sind tendenziell etwas kürzer — zufriedene Kunden fassen sich oft kurz.
Negative Reviews sind häufiger länger, weil unzufriedene Kunden ihre Probleme detaillierter beschreiben.

## 5.4 Frage 3: Welche Wörter verraten die Stimmung?

In [ ]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover
from pyspark.sql.functions import explode

tokenizer = Tokenizer(inputCol="text", outputCol="tokens")
df_tok = tokenizer.transform(df_clean)
remover = StopWordsRemover(inputCol="tokens", outputCol="filtered")
df_filt = remover.transform(df_tok)
df_words = df_filt.select("label", explode("filtered").alias("word"))

top_pos = df_words.filter(col("label") == 1) \
    .groupBy("word").agg(spark_count("*").alias("count")) \
    .orderBy(col("count").desc()).limit(20).toPandas()

top_neg = df_words.filter(col("label") == 0) \
    .groupBy("word").agg(spark_count("*").alias("count")) \
    .orderBy(col("count").desc()).limit(20).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].barh(top_pos["word"][::-1], top_pos["count"][::-1], color="#5DCAA5", edgecolor="#0F6E56")
axes[0].set_title("Top 20 Wörter – POSITIV")
axes[0].set_xlabel("Häufigkeit")
axes[1].barh(top_neg["word"][::-1], top_neg["count"][::-1], color="#F09595", edgecolor="#A32D2D")
axes[1].set_title("Top 20 Wörter – NEGATIV")
axes[1].set_xlabel("Häufigkeit")

plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/03_top_woerter.png", dpi=150, bbox_inches='tight')
plt.show()

**Antwort:** Gemeinsame Wörter wie *book* und *one* erscheinen überall — TF-IDF gewichtet sie herunter.
Die Unterschiede sind aufschlussreich: *great, love, best* bei positiven; *money, bought, product* bei negativen Bewertungen.

## 5.5 Frage 4: Wie sieht die Wortlandschaft aus?

In [ ]:
from wordcloud import WordCloud

pos_freq = dict(zip(top_pos["word"], top_pos["count"]))
neg_freq = dict(zip(top_neg["word"], top_neg["count"]))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

wc_pos = WordCloud(width=800, height=400, background_color="white",
                   colormap="Greens", max_words=80).generate_from_frequencies(pos_freq)
axes[0].imshow(wc_pos, interpolation="bilinear")
axes[0].set_title("Word Cloud – POSITIV", fontsize=16, fontweight='bold')
axes[0].axis("off")

wc_neg = WordCloud(width=800, height=400, background_color="white",
                   colormap="Reds", max_words=80).generate_from_frequencies(neg_freq)
axes[1].imshow(wc_neg, interpolation="bilinear")
axes[1].set_title("Word Cloud – NEGATIV", fontsize=16, fontweight='bold')
axes[1].axis("off")

plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/04_wordcloud.png", dpi=150, bbox_inches='tight')
plt.show()

## 5.6 Frage 5: Wo genau macht das Modell Fehler?

In [ ]:
cm_data = df_preds.groupBy("label", "prediction").count().orderBy("label", "prediction").toPandas()
cm = np.zeros((2, 2), dtype=int)
for _, row in cm_data.iterrows():
    cm[int(row["label"]), int(row["prediction"])] = int(row["count"])

cm_pct = cm / cm.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
labels = ["Negativ (0)", "Positiv (1)"]

im1 = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(labels); axes[0].set_yticklabels(labels)
axes[0].set_xlabel("Vorhersage"); axes[0].set_ylabel("Tatsächlich")
axes[0].set_title("Confusion Matrix (absolut)")
for i in range(2):
    for j in range(2):
        color = "white" if cm[i,j] > cm.max()/2 else "black"
        axes[0].text(j, i, f"{cm[i,j]:,}", ha="center", va="center", fontsize=16, fontweight="bold", color=color)

im2 = axes[1].imshow(cm_pct, cmap="Blues")
axes[1].set_xticks([0, 1]); axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(labels); axes[1].set_yticklabels(labels)
axes[1].set_xlabel("Vorhersage"); axes[1].set_ylabel("Tatsächlich")
axes[1].set_title("Confusion Matrix (prozentual)")
for i in range(2):
    for j in range(2):
        color = "white" if cm_pct[i,j] > cm_pct.max()/2 else "black"
        axes[1].text(j, i, f"{cm_pct[i,j]:.1f}%", ha="center", va="center", fontsize=16, fontweight="bold", color=color)

plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/05_confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

**Antwort:** Das Modell macht symmetrische Fehler — es verwechselt positive und negative
Bewertungen ungefähr gleich häufig (~7–8%). Das zeigt: keine systematische Verzerrung.

## 5.7 Frage 6: Wie sicher ist sich das Modell?

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType
from sklearn.metrics import roc_curve, auc

extract_prob = udf(lambda prob: float(prob[1]), FloatType())
df_roc = df_preds.withColumn("prob_positive", extract_prob(col("probability")))
roc_pd = df_roc.select("label", "prob_positive").toPandas()

fpr, tpr, thresholds = roc_curve(roc_pd["label"], roc_pd["prob_positive"])
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr, tpr, color="#378ADD", lw=2.5, label=f"Logistic Regression (AUC = {roc_auc:.4f})")
ax.plot([0, 1], [0, 1], color="#B4B2A9", lw=1.5, linestyle="--", label="Zufall (AUC = 0.5)")
ax.fill_between(fpr, tpr, alpha=0.15, color="#85B7EB")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC-Kurve – Sentiment-Klassifikation")
ax.legend(loc="lower right", fontsize=12)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/06_roc_kurve.png", dpi=150, bbox_inches='tight')
plt.show()

**Antwort:** Die ROC-Kurve liegt weit über der Zufallslinie. AUC = 0,92 bedeutet:
In 92% der Fälle gibt das Modell einem positiven Review eine höhere Wahrscheinlichkeit
als einem negativen.

## 5.8 Frage 7: Wie verteilen sich die Vorhersage-Wahrscheinlichkeiten?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

neg_probs = roc_pd[roc_pd["label"] == 0]["prob_positive"]
pos_probs = roc_pd[roc_pd["label"] == 1]["prob_positive"]

ax.hist(neg_probs, bins=50, alpha=0.6, color="#F09595", edgecolor="#A32D2D", label="Tatsächlich Negativ")
ax.hist(pos_probs, bins=50, alpha=0.6, color="#5DCAA5", edgecolor="#0F6E56", label="Tatsächlich Positiv")
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1.5, label="Schwellenwert (0.5)")
ax.set_xlabel("Vorhergesagte Wahrscheinlichkeit (positiv)")
ax.set_ylabel("Anzahl")
ax.set_title("Verteilung der Vorhersage-Wahrscheinlichkeiten")
ax.legend()

plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/08_wahrscheinlichkeiten.png", dpi=150, bbox_inches='tight')
plt.show()

**Antwort:** Die beiden Klassen sind deutlich getrennt. Negative Bewertungen
konzentrieren sich links (niedrige Wahrscheinlichkeit für positiv),
positive rechts. Die Überlappung in der Mitte — das sind die schwierigen Fälle,
bei denen auch das Modell unsicher ist.

## 5.9 Metriken-Dashboard

In [ ]:
total = cm.sum()
accuracy = (cm[0,0] + cm[1,1]) / total
precision_neg = cm[0,0] / (cm[0,0] + cm[1,0]) if (cm[0,0] + cm[1,0]) > 0 else 0
precision_pos = cm[1,1] / (cm[0,1] + cm[1,1]) if (cm[0,1] + cm[1,1]) > 0 else 0
recall_neg = cm[0,0] / (cm[0,0] + cm[0,1]) if (cm[0,0] + cm[0,1]) > 0 else 0
recall_pos = cm[1,1] / (cm[1,0] + cm[1,1]) if (cm[1,0] + cm[1,1]) > 0 else 0
f1_neg = 2 * precision_neg * recall_neg / (precision_neg + recall_neg) if (precision_neg + recall_neg) > 0 else 0
f1_pos = 2 * precision_pos * recall_pos / (precision_pos + recall_pos) if (precision_pos + recall_pos) > 0 else 0

metrics = {
    'Accuracy': accuracy, 'AUC-ROC': roc_auc,
    'Precision (neg)': precision_neg, 'Precision (pos)': precision_pos,
    'Recall (neg)': recall_neg, 'Recall (pos)': recall_pos,
    'F1 (neg)': f1_neg, 'F1 (pos)': f1_pos
}

fig, ax = plt.subplots(figsize=(10, 5))
names = list(metrics.keys())
values = list(metrics.values())
bar_colors = ["#378ADD", "#378ADD", "#F09595", "#5DCAA5", "#F09595", "#5DCAA5", "#F09595", "#5DCAA5"]
bars = ax.barh(names[::-1], values[::-1], color=bar_colors[::-1], edgecolor="white", height=0.6)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.01, bar.get_y() + bar.get_height()/2., f'{w:.4f}', ha='left', va='center', fontsize=11, fontweight='bold')
ax.set_xlim([0, 1.15])
ax.set_title("Evaluations-Metriken – Logistic Regression")
ax.axvline(x=0.5, color="#B4B2A9", linestyle="--", alpha=0.5)
ax.grid(True, axis='x', alpha=0.2)

plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/07_metriken_dashboard.png", dpi=150, bbox_inches='tight')
plt.show()

## 5.10 Gespeicherte Dateien

In [ ]:
import os

output_dir = "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output"
print("=" * 55)
print("  GESPEICHERTE DATEIEN")
print("=" * 55)
for f in sorted(os.listdir(output_dir)):
    filepath = os.path.join(output_dir, f)
    size_mb = os.path.getsize(filepath) / (1024*1024) if os.path.isfile(filepath) else 0
    if os.path.isfile(filepath):
        print(f"  {f:45s} {size_mb:>6.2f} MB")
    else:
        print(f"  {f:45s} [Ordner]")
print("=" * 55)

## Fazit: Die Antwort auf unsere Frage

### Kann eine Maschine Emotionen lesen?

**Ja — mit Einschränkungen.**

Mit einem einfachen Bag-of-Words-Ansatz (TF-IDF) und Logistic Regression erreichen wir
**84,7% Accuracy** und eine **AUC von 0,92**. Das bedeutet: In 5 von 6 Fällen erkennt
die Maschine korrekt, ob ein Kunde zufrieden oder unzufrieden ist.

### Was das Modell gelernt hat

Die Maschine hat die **Sprache der Emotionen** gelernt:
- *great, love, best, well* → Signale für Zufriedenheit
- *money, bought, product, work* → Signale für Unzufriedenheit

### Wo die Grenzen liegen

Die ~15% Fehler entstehen bei:
- **Sarkasmus:** *"Oh great, another broken product"* enthält *great* → falsch positiv
- **Gemischte Bewertungen:** *"Tolles Produkt, aber schrecklicher Versand"*
- **Kontextabhängigkeit:** *"Not good"* enthält *good* → Bag-of-Words fehlt der Kontext

### Mögliche Verbesserungen

| Ansatz | Erwartete Verbesserung |
|--------|----------------------|
| N-Gramme (Bi-/Trigramme) | *"not good"* als eigenes Feature |
| Word2Vec / GloVe | Semantische Ähnlichkeiten (toll ≈ super) |
| Deep Learning (LSTM, BERT) | Satzstruktur und Kontext verstehen |

---

*Projekt abgeschlossen. 400.000 Bewertungen geladen, bereinigt, vektorisiert,
klassifiziert und visualisiert — vollständig mit Apache Spark und Python.*